# MLX Incremental Causal Discovery — Full Experiment

This notebook reproduces every result in the README:

1. Synthetic drifting data generation
2. A single end-to-end run with drift detection
3. Main experiment: 5 scenarios × 10 seeds × 4 baselines
4. Signal ablation A1–A6
5. Update scope ablation U1–U4
6. CUSUM sensitivity sweep
7. Aggregation and figures

Runtime on a MacBook Pro M3 Pro (18 GB):
- main experiment: ~13 min
- signal ablation: ~60 min
- update ablation: ~50 min
- CUSUM sweep: ~40 min

To skip long steps, set `QUICK = True` below.

In [5]:
import sys
from pathlib import Path

# Add project root to sys.path so `data`, `experiments`, etc. can be imported
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.synthetic import generate_drifting_data
from online_causal import OnlineCausalDiscovery
from experiments.plots import (
    fig_shd_timeseries, fig_drift_score, fig_update_speedup,
    fig_fallback_rate, fig_delay_fpr,
    fig_ablation_signals, fig_ablation_update,
)

QUICK = False  # set True to use fewer seeds for a smoke test
SEEDS = 2 if QUICK else 10
ROOT = Path('../results')
FIGS = ROOT / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
print("Setup done. Output dir:", FIGS.resolve())

Project root: /Users/quanpin/code/drift
Setup done. Output dir: /Users/quanpin/code/drift/results/figures


## 1. Synthetic drifting data

Generate one dataset per scenario. Each has 5000 samples, 8 variables,
and a single drift injected at sample 2500 (window 5 of size 500).

In [6]:
SCENARIOS = {
    'S1': dict(drift_type='structure',    drift_mode='abrupt'),
    'S2': dict(drift_type='coefficients', drift_mode='abrupt'),
    'S3': dict(drift_type='both',         drift_mode='abrupt'),
    'S4': dict(drift_type='structure',    drift_mode='gradual'),
    'S5': dict(drift_type='coefficients', drift_mode='gradual'),
}

data_demo, meta_demo = generate_drifting_data(
    n_samples=5000, n_vars=8, max_lag=2,
    drift_at=2500, drift_type='both', drift_mode='abrupt',
    random_state=42,
)
print("data shape:", data_demo.shape)
print("drift window (true):", meta_demo['drift_window'])
print("finite:", np.isfinite(data_demo).all())

data shape: (5000, 8)
drift window (true): 5
finite: True


## 2. Single end-to-end run

Run `OnlineCausalDiscovery` on S3 (structure + coefficient drift) and
inspect the per-window detection results.

In [7]:
ocd = OnlineCausalDiscovery(
    window_size=500, max_lag=2, warmup=3,
    kappa=5.0, n_perm=200, z_threshold=2.5,
    random_state=42,
)
results = ocd.run(data_demo)

print(f"{'win':>4} {'score':>8} {'mode':>14} {'reason':>22} changed")
for r in results['records']:
    upd = r.update
    mode = upd.mode if upd else 'none'
    reason = upd.reason if upd else ''
    changed = upd.changed_vars if upd else []
    print(f"{r.index:>4} {r.drift_score:>8.3f} {mode:>14} {reason:>22} {changed}")

print("\nDrift points:", results['drift_points'])

 win    score           mode                 reason changed
   0    0.000           none                        []
   1    0.000           none                        []
   2    0.000           none                        []
   3    1.349           none                        []
   4    1.588           none                        []
   5   16.183    incremental                     ok [3, 4, 5]
   6    6.101    incremental                     ok [3, 4, 5]
   7    6.280    incremental                     ok [5, 7]
   8    3.228           none                        []
   9    1.428           none                        []

Drift points: [5, 6, 7]


## 3. Main experiment

5 scenarios × N seeds × 4 baselines. If the CSV already exists, we
reuse it; otherwise run the script.

In [8]:
main_csv = ROOT / 'main.csv'
if not main_csv.exists():
    print("Running main experiment (may take a few minutes)...")
    import subprocess
    subprocess.run(
        ['python', '-m', 'experiments.run_main',
         '--out', str(ROOT), '--seeds', str(SEEDS)],
        check=True,
    )
else:
    print("Using cached:", main_csv)

df_main = pd.read_csv(main_csv)
print("rows:", len(df_main))

Using cached: ../results/main.csv
rows: 2000


In [9]:
summary_overall = pd.read_csv(ROOT / 'summary_overall.csv')
summary_overall

,baseline,shd_pre_mean,shd_post_mean,f1_post_mean,coef_mse_post_mean,update_time_drift_mean,retrain_time_drift_mean,fallback_rate_drift,update_vs_retrain_ratio,det_precision,det_recall,det_f1
0,B1,9.572,12.128,0.741787,0.132802,0.000000,0.083688,0.0000,1.000000,0.000000,0.00,0.000000
1,B2,9.572,12.128,0.741787,0.132802,0.000000,0.078383,0.0000,NaN,0.603333,0.72,0.630667
2,B3,9.480,12.456,0.728743,0.146919,0.000000,0.000000,0.0000,1.000000,0.000000,0.00,0.000000
3,B4,9.572,12.248,0.742387,0.133512,0.001327,0.078221,0.0125,61.789561,0.660000,0.71,0.668000


## 4. Signal ablation A1–A6

In [10]:
signal_csv = ROOT / 'ablation_signals' / 'summary_by_scenario.csv'
if not signal_csv.exists():
    print("Running signal ablation (long, ~60 min)...")
    import subprocess
    subprocess.run(
        ['python', '-m', 'experiments.run_ablation_signals',
         '--out', str(ROOT / 'ablation_signals'),
         '--seeds', str(SEEDS),
         '--modes', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6'],
        check=True,
    )
    subprocess.run(
        ['python', '-m', 'experiments.aggregate_ablation',
         '--main', str(ROOT / 'ablation_signals' / 'ablation_signals.csv'),
         '--out', str(ROOT / 'ablation_signals')],
        check=True,
    )
else:
    print("Using cached:", signal_csv)

sig_summary = pd.read_csv(signal_csv)
sig_overall = pd.read_csv(ROOT / 'ablation_signals' / 'summary_overall.csv')
sig_overall

Using cached: ../results/ablation_signals/summary_by_scenario.csv


,mode_id,signal_mode,det_precision,det_recall,det_f1,shd_post_mean,f1_post_mean,n_drift_confirmed,fallback_rate_drift,update_vs_retrain_ratio
0,A1,ed_only,0.360000,0.38,0.360000,12.108,0.742617,65,0.000000,69.232432
1,A2,db_only,0.743333,0.82,0.756000,12.216,0.741806,71,0.011111,84.568606
2,A3,shd_only,0.613333,0.68,0.622667,12.268,0.741838,95,0.018182,55.986385
3,A4,ed_db,0.610000,0.69,0.634667,12.208,0.742150,115,0.008333,71.142053
4,A5,all,0.660000,0.71,0.668000,12.248,0.742387,158,0.012500,60.573546
5,A6,db_shd,0.723333,0.79,0.730667,12.252,0.742547,116,0.016667,54.661855


## 5. Update scope ablation U1–U4

In [11]:
update_csv = ROOT / 'ablation_update' / 'summary_by_scenario.csv'
if not update_csv.exists():
    print("Running update scope ablation (long, ~50 min)...")
    import subprocess
    subprocess.run(
        ['python', '-m', 'experiments.run_ablation_update',
         '--out', str(ROOT / 'ablation_update'),
         '--seeds', str(SEEDS)],
        check=True,
    )
    subprocess.run(
        ['python', '-m', 'experiments.aggregate_ablation',
         '--main', str(ROOT / 'ablation_update' / 'ablation_update.csv'),
         '--out', str(ROOT / 'ablation_update')],
        check=True,
    )
else:
    print("Using cached:", update_csv)

upd_summary = pd.read_csv(update_csv)
upd_overall = pd.read_csv(ROOT / 'ablation_update' / 'summary_overall.csv')
upd_overall

Using cached: ../results/ablation_update/summary_by_scenario.csv


,mode_id,mode_label,det_precision,det_recall,det_f1,shd_post_mean,f1_post_mean,n_drift_confirmed,update_time_drift_mean,retrain_time_drift_mean,fallback_rate_drift,update_vs_retrain_ratio,n_seeds
0,U1,seeds,0.640000,0.71,0.654667,12.248,0.742299,157,0.001295,0.077448,0.0125,63.117141,10.0
1,U2,parents,0.640000,0.71,0.654667,12.252,0.742203,157,0.001302,0.077296,0.0125,62.276159,10.0
2,U3,mb,0.660000,0.71,0.668000,12.248,0.742387,158,0.001336,0.077355,0.0125,60.565125,10.0
3,U4,full,0.603333,0.72,0.630667,12.128,0.741787,126,0.000000,0.077845,1.0000,NaN,10.0


## 6. CUSUM sensitivity sweep

Sweep kappa and record delay vs false positive rate per scenario.

In [12]:
sweep_csv = ROOT / 'cusum_sweep.csv'
if not sweep_csv.exists():
    print("Running CUSUM sweep (long, ~40 min)...")
    import subprocess
    subprocess.run(
        ['python', '-m', 'experiments.run_cusum_sweep',
         '--out', str(ROOT), '--seeds', str(SEEDS)],
        check=True,
    )
else:
    print("Using cached:", sweep_csv)

sweep = pd.read_csv(sweep_csv)
sweep.head()

Using cached: ../results/cusum_sweep.csv


,scenario,kappa,delay_mean,delay_std,missed,n_seeds,fpr_mean,fpr_std,precision_mean,recall_mean,f1_mean
0,S1,2.0,0.000,0.000000,0,10,0.633333,0.256038,0.247619,1.00,0.382143
1,S2,2.0,0.100,0.300000,0,10,0.566667,0.238048,0.278333,0.90,0.390476
2,S3,2.0,0.000,0.000000,0,10,0.583333,0.271314,0.305000,1.00,0.425714
3,S4,2.0,0.100,0.300000,0,10,0.620000,0.260000,0.407143,0.95,0.554603
4,S5,2.0,0.375,0.484123,2,10,0.480000,0.256125,0.296667,0.65,0.404286


## 7. Figures

Regenerate all figures from the cached CSVs.

In [13]:
df_main = pd.read_csv(ROOT / 'main.csv')

fig_shd_timeseries(df_main, FIGS)
fig_drift_score(df_main, FIGS)
fig_update_speedup(df_main, FIGS)
fig_fallback_rate(df_main, FIGS)
fig_delay_fpr(ROOT, FIGS)
fig_ablation_signals(ROOT, FIGS)
fig_ablation_update(ROOT, FIGS)

print("Figures written to:", FIGS)
for f in sorted(FIGS.glob('*.png')):
    print(" ", f.name)

Figures written to: ../results/figures
  fig_ablation_signals.png
  fig_ablation_update.png
  fig_delay_fpr.png
  fig_drift_score.png
  fig_fallback_rate.png
  fig_shd_timeseries.png
  fig_update_speedup.png


## 8. Key findings

From the tables and figures above:

### Main experiment
- B4 (ours) matches B1 (full retrain) on SHD / F1 / coefficient MSE
  while running 60–70× faster on drift windows.
- Fallback rate is ~2%, well below the 10% target.
- Detection recall is 0.72 at kappa=5; the residual 0.28 comes from
  gradual coefficient drift (S5), which is the hardest case.

### Signal ablation (A1–A6)
- ΔB alone (A2) gives the highest detection F1 (0.76).
- Adding ED to ΔB (A4) *reduces* F1 to 0.63. ED has a heavy right tail
  in quiet windows, and `max` fusion inflates the CUSUM threshold.
- SHD is complementary to ΔB but weaker on coefficient drift.
- Recommendation: use A2 or A6, not the three-signal default A5.

### Update scope ablation (U1–U4)
- U1, U2, U3 produce nearly identical SHD (12.248 / 12.252 / 12.248).
- U3 gives the highest detection F1 (0.668 vs 0.655 for U1/U2 and
  0.631 for U4).
- U4 gives the best SHD (12.128) but no incremental speedup.
- Local re-learning dominates full retraining on cost (60–63× faster).

### CUSUM sweep
- Small kappa → low delay but high FPR.
- Large kappa → low FPR but higher delay or missed detections.
- Balanced default: kappa ≈ 5.

### Limitations
- At 8 variables, SHD noise floor (≈ 10) dominates the drift signal
  for gradual structure drift (S4). The frozen baseline can match or
  beat retraining in that scenario.
- Coefficient-only gradual drift (S5) remains difficult: recall ≈ 0.5
  even at the best signal combination.